# 📉➡️📈 BIST Derin Değer & Kontraryan Dip Avcısı

**Momentum kovalamayan, "aşırı ucuz kalmış ama çürük olmayan" hisseyi avlayan tarama.**

> *30 yıllık BIST tüccarı mantığı: "Malı ucuza almak kazandırır — ama sadece mal sağlamsa.
> Ucuz + çürük = değer tuzağı. Bütün maharet, ucuz-ve-sağlam ile ucuz-çünkü-batıyor'u ayırmakta."*

---

## Neden bu strateji? (Felsefe)

Bu depodaki mevcut bot (`src/scanner.py`, `src/model.py`) bir **momentum/trend-takip**
sistemidir: ADX>20, EMA hizası, yukarı kırılım, XGBoost yön tahmini. Yani **yükselişi kovalar**.

Bu notebook bunun **tam tersi** felsefeyi kurgular:

| | Momentum botu (mevcut) | Bu tarama (kontraryan derin değer) |
|---|---|---|
| Ne arar? | Yükselen, ivmeli hisse | Dövülmüş, aşırı satımda, terk edilmiş hisse |
| Ne zaman alır? | Kırılımda, kalabalıkla | Panikte, kalabalığın tersine, kademeli |
| Risk | Zirveye yakın alım | "Düşen bıçak" (kademeli alım + stop ile yönetilir) |
| Kâr mantığı | Trend devam eder | Ortalamaya dönüş + değerin fiyatı yakalaması |

**Ana belirleyici = TEKNİK göstergeler.** Bir hissenin ne kadar "aşırı ucuz" olduğunu teknik
ölçer ve sıralar. **Banker (akıllı para) + temel göstergeler ise ÜSTÜNE ekstra puanlama**
katmanıdır — çürük malı eler, sağlam olanı öne çıkarır. Alım tarafında ise **Fibonacci ile
3 kademeli alım** planı kurulur.


## 🏗️ Mimari — nasıl puanlıyoruz?

```
        ┌──────────────────── ÖNCELİKLİ ÇEKİRDEK ────────────────────┐
   OHLCV│  TEKNİK UCUZLUK (0-100)  ×0.60                              │
   ───► │   52h dip konumu·RSI(g+h)·drawdown·Bollinger%B·200EMA·W%R   │──► ÇEKİRDEK
   Hacim│                            +                                │    = sıralamayı
   ───► │  BANKER / AKILLI PARA (0-100)  ×0.40                        │      bunlar belirler
        │   CMF·MFI·A-D·OBV·pozitif diverjans (birikim)              │    + "aşırı ucuz"
        └────────────────────────────────────────────────────────────┘      KAPISI: teknik≥55
                                     ×
        ┌────────────────────────────────────────────────────────────┐
   İş Y.│  TEMEL DEĞER & KALİTE → yalnızca PUANLAMA KATKISI (±%20)     │
   / yf │   PD/DD·EV/EBITDA·F/K·DCF güvenlik marjı·owner earnings      │
   ───► │   temel_çarpan = 1 + 0.20×(temel−50)/50   (≈0.80–1.20)      │
        └────────────────────────────────────────────────────────────┘
                                     ×
        ┌────────────────────────────────────────────────────────────┐
   Temel│  VALUE-TRAP CEZASI (çarpan 0.3×–0.8× / diskalifiye)          │
   ───► │   nakit yakma·zarar·DCF pahalı·likidite·"ucuz çünkü batıyor" │
        └────────────────────────────────────────────────────────────┘
                                     ↓
   NİHAİ = ÇEKİRDEK(teknik+banker) × temel_çarpan × trap_çarpanı   (yüksek = al)
                                     ↓
              Seçilenlere → FIBONACCI 3-KADEME ALIM MERDİVENİ
```

**Öncelik sırası (kullanıcı kurgusu):** Sıralamayı **teknik ucuzluk + banker (akıllı para)**
belirler — ikisi öncelikli çekirdektir. **Temel kriterler yalnızca ölçülü bir puanlama
katkısı** (±%20) yapar; sağlam temel skoru yukarı, zayıf temel aşağı çeker ama çekirdeği
domine etmez. **Value-trap** bayrakları ise çürük malı ağır cezalar ya da tamamen eler.
Böylece en çok dövülmüş hisse otomatik "al" olmaz — hem teknik ucuz, hem akıllı para
toplamış, hem de tuzak olmayan isim öne çıkar.


## 1) Kurulum ve modüller

Bu notebook depodaki `src/deep_value.py` (saf puanlama motoru) ve `src/deep_value_data.py`
(veri adapteri) modüllerini kullanır. Colab'da deposu klonlayıp bu notebook'u kök dizinde
çalıştırın. Veri kaynağı zinciri: **tvdatafeed (rongardF) → yfinance**; temel oranlar:
**İş Yatırım → yfinance**.

In [ ]:
# Colab / yerel kurulum
import sys, subprocess, os
from pathlib import Path

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# --- Bağımlılıklar ---
try:
    import pandas, numpy, matplotlib, requests, openpyxl  # noqa
except ImportError:
    _pip("pandas", "numpy", "matplotlib", "requests", "openpyxl")
# tvdatafeed (rongardF fork) — TradingView verisi; yoksa yfinance yedeğe düşülür
try:
    import tvDatafeed  # noqa
except ImportError:
    _pip("git+https://github.com/rongardF/tvdatafeed.git", "yfinance")

# --- src/ modüllerini bul; Colab'da yalnızca .ipynb yüklenmişse depoyu klonla ---
REPO_URL = "https://github.com/skumova/bist-100-teknik-analiz-ve-filtreleme-i-yi.ipynb.git"
REPO_DIR = "bist-100-teknik-analiz-ve-filtreleme-i-yi.ipynb"
BRANCH   = "claude/bist-technical-analysis-ol92sk"

def _has_src(p):  # içinde src/deep_value.py olan kök mü?
    return (Path(p) / "src" / "deep_value.py").exists()

ROOT = None
for cand in [os.getcwd(), *map(str, Path(os.getcwd()).parents)]:
    if _has_src(cand):
        ROOT = cand; break

if ROOT is None:  # bulunamadı → depoyu klonla
    if not _has_src(REPO_DIR):
        print("src/ bulunamadı, depo klonlanıyor…")
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=False)
    if _has_src(REPO_DIR):
        ROOT = os.path.abspath(REPO_DIR)

if ROOT is None:
    raise RuntimeError(
        "src/ modülleri bulunamadı. Depo özel (private) ise Colab'da şu şekilde klonlayın:\n"
        "  !git clone -b %s https://<TOKEN>@github.com/skumova/"
        "bist-100-teknik-analiz-ve-filtreleme-i-yi.ipynb.git\n"
        "Ardından o klasöre girip bu notebook'u tekrar çalıştırın." % BRANCH)

os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import importlib
from src import deep_value as dv
from src import deep_value_data as dvd
importlib.reload(dv); importlib.reload(dvd)
print("Repo kökü:", ROOT)
print("Motor yüklendi. TECH_GATE =", dv.TECH_GATE, "| BONUS_STRENGTH =", dv.BONUS_STRENGTH)

## 2) Teknik Ucuzluk — öncelikli çekirdeğin 1. bileşeni (×0.60, 0-100)

100 = maksimum dövülmüş / aşırı satımda. Trend-takibin **tersine**: düşük RSI, dipteki konum,
derin drawdown **yüksek** puan alır. Alt bileşenler ve ağırlıkları:

| Bileşen | Ağırlık | Mantık |
|---|---|---|
| **52-hafta banttaki konum** | %28 | Dibe yakınlık — "aşırı ucuz"un kalbi. `(fiyat−52hDip)/(52hTepe−52hDip)` |
| **Günlük RSI(14)** | %20 | Aşırı satım (RSI 50→0 puan, ≤15→100) |
| **Haftalık RSI(14)** | %12 | Kalıcı ucuzluk teyidi (gürültüye dayanıklı) |
| **52-hafta zirveden düşüş** | %15 | Ne kadar dövülmüş (%10→0, %60→100) |
| **Bollinger %B** | %10 | Alt banda/altına sarkma |
| **200-EMA sapması** | %10 | Uzun vade ortalamanın ne kadar altında |
| **Williams %R** | %5 | Ek aşırı-satım teyidi |

Ayrıca **dip teyidi**: MACD histogramı son barlarda yukarı dönüyorsa (düşüş ivmesi kırıldı)
`dip_confirm=True`. Kapı: `teknik ≥ 55` → "aşırı ucuz aday".


## 3) BANKER / Akıllı Para — öncelikli çekirdeğin 2. bileşeni (×0.40)

**Teknik ile birlikte önceliklidir**: sıralamayı bu ikisi belirler. Aşırı ucuz bir hissede
asıl aradığımız, fiyat dibe yatarken **güçlü ellerin sessizce topladığının** izidir. Klasik
dip-avı sinyali **pozitif diverjans**tır: fiyat düşük/yatay ama para akışı yukarı.

- **CMF (Chaikin Money Flow)** > 0 → birikim (banker topluyor)
- **MFI (Money Flow Index)** → hacim ağırlıklı aşırı-satım/dönüş
- **A/D çizgisi & OBV eğimi** → kümülatif hacim baskısı yönü
- **Pozitif diverjans bonusu** → fiyat son 20 günde düşmüş **ama** A/D/CMF yukarı

Banker skoru **düşükse** hisse hâlâ *dağıtım* (satış) altındadır → "ucuz ama akıllı para
girmemiş, acele etme / daha derin kademeleri bekle". Yüksekse teknik ucuzluk daha güvenilir.


## 4) Temel Değer & Kalite — yalnızca puanlama KATKISI (±%20) + Value-Trap elemesi

Temel kriterler çekirdeği (teknik+banker) **domine etmez**; sadece ±%20'lik bir çarpanla
ince ayar yapar (`temel_çarpan = 1 + 0.20×(temel−50)/50`). İki alt blok **geometrik**
birleştirilir (ikisi de gerekli):

- **Ucuzluk (value):** PD/DD, EV/EBITDA, F/K, EV/Satış — sektör-göreli ve mutlak.
  *Eksiğe dayanıklı*: BIST'te İş Yatırım bazı oranları vermez, Yahoo trailing F/K çevrimsel
  diplerde 500+ olabilir → eldeki metriklerin ağırlıklı ortalaması alınır, tek metrik skoru
  sıfırlamaz.
- **Kalite (quality):** owner earnings (gerçek nakit üretimi), DCF güvenlik marjı, Buffett
  skoru, borçluluk, ROE.

**Value-trap bayrakları** (30 yıllık trader refleksi) nihai skoru çarpanla cezalandırır:

| Bayrak | Ceza | Örnek (2026-07-03) |
|---|---|---|
| Nakit yakıyor (negatif owner earnings) | ×0.30 | ECILC (OE −1.305M) |
| Zarar (negatif net kâr) | ×0.55 | ECILC (−924M) |
| DCF'e göre pahalı + AVOID | ×0.55 | EREGL (marj −%1175), ARCLK (−%127) |
| Ucuz çünkü zarar (F/K yok + düşük PD/DD + nakit yok) | ×0.50 | CANTE (PD/DD 0.4, OE=NaN) |
| İşletme değeri pahalı (EV/EBITDA>15) | ×0.75 | SASA (EV/EBITDA 26) |
| Likidite tuzağı / düşen bıçak | ×0.60 / ×0.80 | — |

Nakit yakan **+** zarar eden **+** AVOID kombinasyonu → **tam diskalifiye**.


## 5) Alım tarafı — Fibonacci 3-Kademeli Merdiven

"Düşen bıçağı" tek hamlede tutmayız. Seçilen her hisse için son ~180 barın baskın swing'i
(tepe **H**, dip **L**) alınır; H→L düşüşünün Fibonacci seviyeleri referanstır. Alım
kademeleri **yalnızca güncel fiyatın altındaki** seviyelere konur ve **derine daha çok
ağırlık** verilir (ucuzsa daha çok al):

- **1. kademe (%25)** — güncel fiyat / ilk Fib desteği (0.618–1.0)
- **2. kademe (%35)** — 1.272 uzantısı (kapitülasyon)
- **3. kademe (%40)** — 1.618 uzantısı (aşırı kapitülasyon)
- **HARD STOP** = en derin kademe − 1.5×ATR (bu da tutmazsa tez yanlış)

DCF içsel değeri varsa merdivenin ağırlıklı ortalama maliyetine göre **beklenen getiri**
hesaplanır (teknik giriş ↔ temel hedef köprüsü).


## 6) Universe ve tarama — **TÜM BIST**

Evren, `src.scanner.fetch_bist_symbols()` ile **TradingView Scanner API'sinden tüm BIST
hisseleri** (canlı, ~500+ sembol) olarak çekilir; API'ye ulaşılamazsa HTML tablo yedeklerine,
en sonda `config.BIST100_SYMBOLS` çekirdek listesine düşer.

**Ana belirleyici teknik olduğu için** önce her hissenin teknik ucuzluğu hesaplanır; kapıyı
(TECH_GATE) geçemeyecek kadar pahalı olanlara **temel/banker verisi çekilmez** — yüzlerce
hissede gereksiz API isteği yapılmaz, tarama makul sürede biter. Yine de tüm evren teknik
olarak taranır.

In [ ]:
from src import config
from src.scanner import fetch_bist_symbols

# TÜM BIST evreni (canlı; başarısızsa yedeklere düşer)
SYMBOLS = fetch_bist_symbols()
print(f"Evren: {len(SYMBOLS)} BIST hissesi taranacak.")
# Hızlı deneme için küçük alt küme kullanmak isterseniz:
# SYMBOLS = ["ULKER","PETKM","TURSG","MAVI","FROTO","TUKAS","SASA","CANTE","ECILC","EREGL","ARCLK","GARAN"]

report, results = dvd.screen_universe(
    SYMBOLS,
    n_bars=600,           # ~2.5 yıl günlük bar (200EMA + 52h için yeterli)
    fib_lookback=180,     # Fibonacci swing penceresi
    tech_prefilter=True,  # teknik kapıyı geçemeyene temel veri çekme (tüm BIST için şart)
    with_ladder_top_n=20, # en iyi 20 adaya Fib merdiveni hesapla
)
print(f"\n{len(results)} hisse tarandı, {int(report['asiri_ucuz'].sum())} tanesi teknik kapıyı geçti.\n")
report.head(30)

### 6a) Sadece "aşırı ucuz" adaylar (kapıyı geçenler, tuzaksızlar önde)

In [ ]:
import pandas as pd
pd.set_option("display.width", 220, "display.max_columns", 30)

adaylar = report[report["asiri_ucuz"] & (~report["disqualified"])].copy()
cols = ["symbol","sector","final_score","cekirdek","tech_ucuzluk","banker","temel_skor",
        "temel_carpan","value","quality","rsi_d","rsi_w","pos_52w","drawdown","birikim","trap_mult","trap_flags"]
adaylar[cols].reset_index(drop=True)

### 6b) Değer tuzağı olarak elenenler (ders niteliğinde)

Bunlar teknik olarak ucuz **ama** temelde çürük — sistem bilerek geri iter. En çok dövülmüş
hissenin en iyi alım OLMADIĞINI gösterir.

In [ ]:
tuzaklar = report[(report["trap_flags"] != "") | (report["disqualified"])].copy()
tuzaklar[["symbol","final_score","cekirdek","tech_ucuzluk","banker","temel_skor","value","quality",
          "trap_mult","disqualified","trap_flags"]].reset_index(drop=True)

## 7) Seçilen hisse için Fibonacci 3-kademe alım grafiği

In [ ]:
import matplotlib.pyplot as plt

def plot_fib_ladder(symbol, results, n_bars=250):
    res = next((r for r in results if r.symbol == symbol), None)
    if res is None or res.ladder is None:
        print(f"{symbol}: sonuç/merdiven yok."); return
    df = dvd.load_daily_ohlcv(symbol, n_bars=600).tail(n_bars)
    L = res.ladder
    fig, ax = plt.subplots(figsize=(13, 6))
    ax.plot(df.index, df["close"], color="#1f2d3d", lw=1.3, label="Kapanış")
    ax.axhline(L.swing_high, color="#888", ls="--", lw=0.8)
    ax.text(df.index[0], L.swing_high, f"  swing tepe {L.swing_high}", va="bottom", color="#888", fontsize=8)
    palette = ["#2e7d32", "#f9a825", "#c62828"]
    for i, rung in enumerate(L.rungs):
        c = palette[min(i, 2)]
        ax.axhline(rung["price"], color=c, lw=1.4, alpha=0.9)
        ax.text(df.index[-1], rung["price"],
                f"  {i+1}. kademe %{rung['weight_pct']} @ {rung['price']}",
                va="center", color=c, fontsize=9, fontweight="bold")
    ax.axhline(L.hard_stop, color="#6a1b9a", ls=":", lw=1.2)
    ax.text(df.index[-1], L.hard_stop, f"  STOP {L.hard_stop}", va="center", color="#6a1b9a", fontsize=8)
    title = (f"{symbol} — Nihai {res.final_score} | Teknik {res.technical.total} | "
             f"Banker {res.banker.total} | Temel {res.fundamental.total}")
    if L.expected_upside_pct is not None:
        title += f"\nDCF hedef {L.dcf_target} → merdiven ort. maliyetine göre bek. getiri %{L.expected_upside_pct}"
    ax.set_title(title, fontsize=11)
    ax.legend(loc="upper right"); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()

# En iyi aday için çiz:
en_iyi = report[report["asiri_ucuz"] & (~report["disqualified"])]
if not en_iyi.empty:
    plot_fib_ladder(en_iyi.iloc[0]["symbol"], results)

In [ ]:
def alim_plani(symbol, results, sermaye=100_000):
    """Bir hisse için kademeli alım planını TL bazında yazdırır."""
    res = next((r for r in results if r.symbol == symbol), None)
    if res is None or res.ladder is None:
        print(f"{symbol}: merdiven yok."); return
    L = res.ladder
    print(f"=== {symbol} — Kademeli Alım Planı (sermaye {sermaye:,.0f} TL) ===")
    print(f"Nihai skor {res.final_score} | Teknik {res.technical.total} | "
          f"Banker {res.banker.total} | Temel {res.fundamental.total}")
    if res.trap.flags:
        print("⚠️  Tuzak bayrakları:", "; ".join(res.trap.flags))
    print(f"Güncel {L.current_price} | swing {L.swing_low}–{L.swing_high} | STOP {L.hard_stop}\n")
    toplam_lot = 0
    for i, r in enumerate(L.rungs, 1):
        pay = sermaye * r["weight_pct"] / 100
        lot = int(pay / r["price"])
        toplam_lot += lot
        print(f"  {i}. kademe @ {r['price']:>8}  (%{r['weight_pct']:<4} = {pay:>10,.0f} TL "
              f"→ ~{lot:,} lot)  [{r['note']}]")
    print(f"\n  Toplam ~{toplam_lot:,} lot | STOP {L.hard_stop} "
          f"(en derin kademeden ~%{(1-L.hard_stop/L.rungs[-1]['price'])*100:.1f} aşağıda)")
    if L.expected_upside_pct is not None:
        print(f"  DCF hedef {L.dcf_target} → beklenen getiri ~%{L.expected_upside_pct}")

if not en_iyi.empty:
    alim_plani(en_iyi.iloc[0]["symbol"], results)

## 8) 📊 Detaylı Excel çıktısı

Tüm taramayı çok-sayfalı, biçimlendirilmiş bir Excel dosyasına aktarır:

| Sayfa | İçerik |
|---|---|
| **Ozet** | Sıralı özet tablo (tüm evren) |
| **Adaylar** | Teknik kapıyı geçen, tuzaksız hisseler — tüm alt bileşenlerle |
| **Tuzaklar** | Value-trap bayraklı / diskalifiye edilenler (neden elendiği) |
| **Detay** | Her hisse için teknik+banker+temel **ham değerler** (RSI, CMF, MFI, 52h, PD/DD…) |
| **Alim_Plani** | Fibonacci 3-kademe merdivenleri (fiyat, ağırlık, TL, ~lot, stop, DCF hedef) |

`final_score` kolonu renk skalasıyla (kırmızı→yeşil) boyanır.

In [ ]:
SERMAYE = 100_000   # Alım planı TL/lot dağılımı için varsayılan sermaye

xlsx_path = dvd.export_to_excel(report, results,
                                path="bist_derin_deger_tarama.xlsx",
                                capital=SERMAYE)
print("Excel oluşturuldu:", xlsx_path)

# Colab'da otomatik indir:
try:
    from google.colab import files
    files.download(xlsx_path)
except Exception:
    print("Yerel çalışma: dosya çalışma dizininde ->", xlsx_path)

## 9) 📸 Canlı örnek çıktı — 2026-07-03 (gerçek veri)

Aşağıdaki değerler, motorun **gerçek İş Yatırım temel verisi + gerçek OHLCV** üzerinde
o günkü çıktısıdır. Notebook'u çalıştırınca güncel tarih için yeniden hesaplanır.

### ULKER — tam gerçek çalıştırma (öncelikli çekirdek örneği)

| Bileşen | Değer | Not |
|---|---|---|
| **Nihai skor** | **58.9** | çekirdek × temel_çarpan × trap |
| Çekirdek (teknik+banker) | 50.8 | `0.60×67.8 + 0.40×25.4` |
| ⤷ Teknik ucuzluk | **67.8** | RSI 30.3, 52h dibinde (pos 0.08), drawdown %29, MACD dip teyidi ✅ |
| ⤷ Banker/akıllı para | **25.4** | CMF −0.12 → **hâlâ dağıtım, akıllı para girmemiş** |
| Temel katkı | ×1.158 | temel 89.5 (value 87 / quality 91) → +%16 katkı |
| Value-trap | ×1.00 | temiz |

> **Öncelik mantığı iş başında:** ULKER temelde mükemmel (89.5) olmasına rağmen **banker
> skoru düşük** olduğu için çekirdek 50.8'e iniyor — sistem "ucuz ve sağlam, ama akıllı para
> daha toplamıyor, acele etme / kademeli gir" diyor. Temel tek başına sıralamayı yukarı
> taşıyamıyor; öncelik teknik+banker'da.

### Aşırı-ucuz aday havuzu (kapıyı geçenler) — gerçek temel + RSI

Nihai sıralama canlı çalıştırmada her hissenin **banker** skoruyla belirlenir; aşağıdaki
temel skorlar (İş Yatırım) ve RSI (o gün) gerçektir:

| Hisse | RSI | Temel | value/quality | Neden aday |
|---|---|---|---|---|
| **ULKER** | 30.3 | 89.5 | 87/91 | 52h dibinde, DCF marjı %68, EV/EBITDA 4.3 |
| **PETKM** | 34.0 | 83.5 | 93/76 | PD/DD 0.7, güvenlik marjı %34, nakit üretiyor |
| **TURSG** | 37.8 | 76.3 | 55/100 | STRONG_BUY, F/K 5.6, güvenlik marjı %64 |
| **MAVI** | 38.9 | 76.7 | 68/84 | EV/EBITDA 2.9, güvenlik marjı %42 |
| **FROTO** | 36.3 | 64.9 | 65/65 | Güçlü hendek, F/K 8.5, ama marj dar %17 |

### Değer tuzağı olarak elenenler (teknik ucuz ama çürük)

| Hisse | RSI | Neden ELENDİ | Çarpan |
|---|---|---|---|
| **CANTE** | **27.4** (en oversold!) | Owner earnings YOK, PD/DD 0.4 → "ucuz çünkü zarar" | ×0.50 |
| **ECILC** | 39.7 | Nakit yakıyor (−1.305M) + zarar (−924M) + AVOID | **diskalifiye** |
| **EREGL** | 30.6 | DCF marjı −%1175 (kâr çökmüş, çevrimsel), AVOID | ×0.55 |
| **ARCLK** | – | DCF marjı −%127, düşük nakit getirisi, AVOID | ×0.55 |

> **Kilit gözlem:** En çok dövülmüş hisse (CANTE, RSI 27) **en iyi alım değil** — sistem onu
> tuzak olarak eler. Asıl alınacak, hem aşırı ucuz **hem** nakit üreten ULKER/PETKM/TURSG.

### ULKER — gerçek 3-kademe Fibonacci merdiveni (güncel 100.1 ₺)

| Kademe | Fiyat | Ağırlık | Fib | Not |
|---|---|---|---|---|
| 1 | **96.5** | %25 | 1.000 | swing dip retesti |
| 2 | **84.2** | %35 | 1.272 | kapitülasyon uzantısı |
| 3 | **68.6** | %40 | 1.618 | aşırı kapitülasyon |
| **STOP** | **64.3** | — | — | en derin kademe − 1.5×ATR |

*52h yüksek 141.7 / düşük 96.5, drawdown %29, MACD dip teyidi ✅. Banker skoru düşük olduğu
için "kademeli gir, tek seferde yükleme" mesajı verir.*


## 10) ⚠️ Risk yönetimi ve kullanım notları

1. **Kademeli gir, aşkla değil planla.** 3 kademe = maliyet ortalaması + yanlışsa kontrollü
   zarar. Banker skoru düşükken (akıllı para girmemişken) ilk kademeyi küçük tut, derin
   kademeleri bekle.
2. **STOP'a sadık kal.** En derin kademe − 1.5×ATR kırılırsa tez yanlıştır; değer tuzağına
   dönmüş olabilir. Zararı büyütme.
3. **Value-trap bayraklarını ciddiye al.** "Ucuz" bir hisse aylarca daha ucuzlayabilir.
   Nakit yakan / zarar eden / DCF'e göre pahalı isimlerden uzak dur (sistem zaten cezalandırır).
4. **Katalizör/likidite.** Derin değer, katalizör olmadan uzun sürebilir. Likidite tuzağı
   bayraklı (düşük TL hacim) isimlerde pozisyon küçük olsun.
5. **Temel veri gecikmeli/eksiktir.** İş Yatırım oranları çeyrekliktir; teyit için son bilanço
   ve KAP'a bak. F/K çevrimsel diplerde yanıltır → PD/DD ve EV/EBITDA'ya ağırlık ver.
6. **Bu bir yatırım tavsiyesi değildir.** Eğitim ve araştırma amaçlı bir tarama aracıdır.
   Kararı kendi analizinle ver.

---

### Parametre ayarları (`src/deep_value.py`)
`TECH_GATE` (aşırı-ucuz eşiği, vars. 55) · `CORE_W_TECH`/`CORE_W_BANKER` (öncelikli çekirdek
ağırlıkları, vars. 0.60/0.40) · `FUND_BONUS_STRENGTH` (temel kriter katkı gücü, ±%20) ·
teknik alt-ağırlıklar `TECH_WEIGHTS` · Fibonacci `_FIB_RATIOS` ve kademe ağırlıkları.
Banker'a daha çok öncelik vermek için `CORE_W_BANKER`'ı artır. Kendi tarzına göre kalibre et.
